# Demo: one nonconvex transcript Ball-DP finite-prior workflow

This lightweight notebook is the pedagogical end-to-end version of the nonconvex transcript experiment. It builds one local finite prior, trains one ERM release, one Ball-DP release, and one standard-DP release, then runs the known-inclusion and unknown-inclusion transcript attacks.

For thesis-scale evidence, use the separate aggregated runner/notebook. This demo is intended for future reference and debugging, not as the final statistical evidence.


## Threat model and transcript likelihoods

The finite-prior setup is the same as in the convex notebook. We fix

$$
S = \{z_1,\ldots,z_m\} \subseteq B(u,r),
\qquad
Z \sim \pi,
\qquad
D_Z = D^- \cup \{Z\}.
$$

The nonconvex difference is the observation. Instead of a final Gaussian-perturbed ERM solution, the adversary observes a DP-SGD transcript. After subtracting known non-target batch contributions, a retained step has residual form

$$
R_t = I_t \, g_t(Z) + \xi_t,
\qquad
\xi_t \sim \mathcal N(0,\sigma_t^2 I).
$$

The known-inclusion attack scores target-present steps. The unknown-inclusion attack uses the Bernoulli mixture likelihood

$$
(1-q_t)\,\mathcal N(0,\sigma_t^2 I)
+
q_t\,\mathcal N(g_t(z_i),\sigma_t^2 I).
$$

The same support $S$, target policy, and baseline $\kappa=\max_i\pi_i$ are used for both observation models.

## Imports

The tutorial uses the public training API plus the canonical finite-prior setup helpers. The attack scorer is transcript-specific, but the trial object is the same object used by the convex tutorial.

In [ ]:
from __future__ import annotations

import dataclasses as dc
import math
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp import (
    ball_rero,
    get_public_curve_history,
    make_finite_identification_prior,
    make_uniform_ball_prior,
)
from quantbayes.ball_dp.api import (
    attack_nonconvex_finite_prior_trial,
    calibrate_ball_sgd_noise_multiplier,
    evaluate_release_classifier,
    extract_privacy_epsilon,
    make_trace_metadata_from_release,
)
from quantbayes.ball_dp.attacks.ball_policy import BallTraceMapAttackConfig
from quantbayes.ball_dp.attacks.finite_prior_setup import (
    CandidateSource,
    find_feasible_replacement_banks,
    make_replacement_trial,
    select_support_from_bank,
    target_positions_for_support,
)
from quantbayes.ball_dp.attacks.gradient_based import (
    DPSGDTraceRecorder,
    subtract_known_batch_gradients,
)
from quantbayes.ball_dp.theorem import (
    TheoremBounds,
    TheoremModelSpec,
    TrainConfig,
    certified_lz,
    fit_release,
    make_model,
)
from quantbayes.ball_dp.types import ArrayDataset

plt.rcParams.update(
    {
        "figure.figsize": (7.4, 4.4),
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    }
)

## Configuration

The defaults below are chosen to be feasible and interpretable: full transcript capture, a nontrivial local radius, and theorem constants that give Ball-DP a real sensitivity advantage over standard DP.


In [ ]:
SEED = 0

# Synthetic embedding data.
N_SAMPLES = 800
N_FEATURES = 16
TEST_SIZE = 0.30
CLASS_SEP = 2.8
LABEL_NOISE = 0.0
EMBEDDING_BOUND = 5.0

# Canonical finite-prior setup.
RADIUS = 2.0
M = 6
SUPPORT_SELECTION = "farthest"

# Nonconvex model and training.
HIDDEN_DIM = 96
A = 5.0
LAMBDA = 5.0
NUM_STEPS = 250
BATCH_SIZE = 128
CLIP_NORM = 50.0
LEARNING_RATE = 1e-2
DELTA = 1e-5
TARGET_EPSILON = 12.0
CAPTURE_EVERY = 1
EVAL_EVERY = 10
CHECKPOINT_SELECTION = "best_public_eval_accuracy"
ORDERS = tuple(range(2, 13))


## Synthetic data

The data are scaled into the public norm bound $B$. In real experiments, this cell is replaced by your real embedding arrays.

In [ ]:
def make_synthetic_embeddings(*, seed: int):
    X, y = make_classification(
        n_samples=N_SAMPLES,
        n_features=N_FEATURES,
        n_informative=N_FEATURES,
        n_redundant=0,
        n_repeated=0,
        n_clusters_per_class=1,
        class_sep=CLASS_SEP,
        flip_y=LABEL_NOISE,
        random_state=seed,
    )
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    X_train, X_public, y_train, y_public = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=seed,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_public = scaler.transform(X_public).astype(np.float32)

    max_norm = max(
        float(np.linalg.norm(X_train, axis=1).max()),
        float(np.linalg.norm(X_public, axis=1).max()),
        1e-12,
    )
    scale = 0.95 * EMBEDDING_BOUND / max_norm
    return (
        (scale * X_train).astype(np.float32),
        y_train.astype(np.int32),
        (scale * X_public).astype(np.float32),
        y_public.astype(np.int32),
    )


X_train, y_train, X_public, y_public = make_synthetic_embeddings(seed=SEED)
FEATURE_DIM = int(X_train.shape[1])
NUM_CLASSES = int(len(np.unique(np.concatenate([y_train, y_public]))))

display(
    pd.DataFrame(
        {
            "split": ["private train", "public/eval"],
            "n": [len(X_train), len(X_public)],
            "feature_dim": [FEATURE_DIM, FEATURE_DIM],
            "max_l2_norm": [
                float(np.linalg.norm(X_train, axis=1).max()),
                float(np.linalg.norm(X_public, axis=1).max()),
            ],
            "class_0": [int(np.sum(y_train == 0)), int(np.sum(y_public == 0))],
            "class_1": [int(np.sum(y_train == 1)), int(np.sum(y_public == 1))],
        }
    )
)

## Model class and certified radius-to-gradient constant

The theorem-side bound uses a public model class and public norm bounds. The library computes the certified Lipschitz quantity $L_z$ used in Ball-SGD sensitivity calibration.

In [ ]:
spec = TheoremModelSpec(
    d_in=int(FEATURE_DIM),
    hidden_dim=int(HIDDEN_DIM),
    task="binary",
    parameterization="dense",
    constraint="op",
)

bounds = TheoremBounds(
    B=float(EMBEDDING_BOUND),
    A=float(A),
    Lambda=float(LAMBDA),
)

lz = float(certified_lz(spec, bounds))

display(
    pd.DataFrame(
        [
            {
                "feature_dim": FEATURE_DIM,
                "hidden_dim": HIDDEN_DIM,
                "B": EMBEDDING_BOUND,
                "A": A,
                "Lambda": LAMBDA,
                "certified_lz": lz,
                "radius": RADIUS,
            }
        ]
    )
)

## Build the canonical finite-prior trial

The support is constructed before the target is chosen. For nonconvex transcript attacks, training is more expensive, so this tutorial samples one hidden target from the support. The same support can also be enumerated in real experiments.

In [ ]:
public_source = CandidateSource("public", X_public, y_public)

bank = find_feasible_replacement_banks(
    X_train=X_train,
    y_train=y_train,
    candidate_sources=[public_source],
    radius=RADIUS,
    min_support_size=M,
    num_banks=1,
    seed=SEED,
    anchor_selection="large_bank",
    strict=True,
)[0]

support = select_support_from_bank(
    bank,
    m=M,
    selection=SUPPORT_SELECTION,
    seed=SEED,
    draw_index=0,
)

target_pos = target_positions_for_support(
    support,
    policy="sample",
    num_targets=1,
    seed=SEED,
)[0]

trial = make_replacement_trial(
    X_train=X_train,
    y_train=y_train,
    support=support,
    target_support_position=int(target_pos),
)

support_df = pd.DataFrame(
    {
        "support_position": np.arange(support.m),
        "source_id": list(support.source_ids),
        "label": support.y,
        "distance_to_center": support.distances_to_center,
        "prior_weight": support.weights,
        "is_target": np.arange(support.m) == int(target_pos),
    }
)

print("center:", support.center_source_id)
print("support hash:", support.support_hash)
print("target:", trial.target_source_id)
print("target index in training dataset:", trial.target_index)
print("oblivious baseline kappa:", support.oblivious_kappa)
display(support_df)

## Support visualization

This plot is a PCA projection of public points, the center $u$, and the support $S$. The true target is highlighted.

In [ ]:
def plot_support_geometry(X_public, y_public, support, target_pos: int):
    pca = PCA(n_components=2, random_state=SEED)
    all_points = np.concatenate(
        [X_public, support.center_x[None, :], support.X],
        axis=0,
    )
    Z = pca.fit_transform(all_points)
    Z_public = Z[: len(X_public)]
    z_center = Z[len(X_public)]
    Z_support = Z[len(X_public) + 1 :]

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for cls in sorted(np.unique(y_public).tolist()):
        mask = y_public == cls
        ax.scatter(
            Z_public[mask, 0],
            Z_public[mask, 1],
            s=18,
            alpha=0.20,
            label=f"public class {int(cls)}",
        )

    ax.scatter(
        [z_center[0]],
        [z_center[1]],
        s=170,
        marker="*",
        edgecolor="black",
        linewidth=0.8,
        label="center u",
        zorder=5,
    )

    ax.scatter(
        Z_support[:, 0],
        Z_support[:, 1],
        s=90,
        marker="o",
        edgecolor="black",
        linewidth=0.7,
        label="support S",
        zorder=4,
    )
    ax.scatter(
        [Z_support[target_pos, 0]],
        [Z_support[target_pos, 1]],
        s=170,
        marker="X",
        edgecolor="black",
        linewidth=0.8,
        label="hidden target",
        zorder=6,
    )

    for i, z in enumerate(Z_support):
        ax.text(z[0], z[1], str(i), fontsize=9, ha="center", va="center")

    ax.set_title("Canonical support and sampled target")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(loc="best")
    plt.show()


plot_support_geometry(X_public, y_public, support, int(target_pos))

## Training helpers

The helpers below train ERM, Ball-DP, and standard-DP releases using the same `TrainConfig` path. The recorder stores the sanitized gradient transcript needed for attacks.

In [ ]:
@dataclass
class ReleaseBundle:
    name: str
    mechanism: str
    release: Any
    noise_multiplier: float
    epsilon_primary: float
    epsilon_ball: float
    epsilon_standard: float
    utility_value: float
    recorder: Any


def evaluate_accuracy_safe(release: Any, X: np.ndarray, y: np.ndarray) -> float:
    try:
        out = evaluate_release_classifier(release, X, y, batch_size=1024)
        if "accuracy" in out:
            return float(out["accuracy"])
    except Exception:
        pass
    return float(release.utility_metrics.get("accuracy", np.nan))


def extract_epsilon_safe(release: Any, accounting_view: str) -> float:
    try:
        return float(extract_privacy_epsilon(release, accounting_view=accounting_view))
    except Exception:
        return float("inf")


def calibrate_noise_multiplier(*, privacy: str) -> float:
    out = calibrate_ball_sgd_noise_multiplier(
        dataset_size=int(len(trial.X_full)),
        radius=float(RADIUS),
        lz=float(lz),
        num_steps=int(NUM_STEPS),
        batch_size=int(BATCH_SIZE),
        clip_norm=float(CLIP_NORM),
        target_epsilon=float(TARGET_EPSILON),
        delta=float(DELTA),
        privacy=str(privacy),
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        orders=ORDERS,
        lower=1e-3,
        upper=0.25,
        max_upper=128.0,
        num_bisection_steps=12,
    )
    return float(out["noise_multiplier"])


def train_recorded_release(
    *,
    name: str,
    mechanism: str,
    privacy: str,
    noise_multiplier: float,
    seed: int,
) -> ReleaseBundle:
    recorder = DPSGDTraceRecorder(
        capture_every=int(CAPTURE_EVERY),
        keep_models=True,
        keep_batch_indices=True,
    )

    model = make_model(
        spec,
        key=jr.PRNGKey(int(seed)),
        init_project=True,
        bounds=bounds,
    )

    cfg = TrainConfig(
        radius=float(RADIUS),
        privacy=str(privacy),
        delta=float(DELTA),
        num_steps=int(NUM_STEPS),
        batch_size=int(BATCH_SIZE),
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        clip_norm=float(CLIP_NORM),
        noise_multiplier=float(noise_multiplier),
        learning_rate=float(LEARNING_RATE),
        checkpoint_selection=str(CHECKPOINT_SELECTION),
        eval_every=int(EVAL_EVERY),
        eval_batch_size=1024,
        normalize_noisy_sum_by="batch_size",
        seed=int(seed),
    )

    release = fit_release(
        model,
        spec,
        bounds,
        np.asarray(trial.X_full, dtype=np.float32),
        np.asarray(trial.y_full, dtype=np.int32),
        X_eval=np.asarray(X_public, dtype=np.float32),
        y_eval=np.asarray(y_public, dtype=np.int32),
        train_cfg=cfg,
        orders=ORDERS,
        trace_recorder=recorder,
        record_operator_norms=True,
        operator_norms_every=max(10, int(NUM_STEPS) // 5),
    )

    eps_ball = extract_epsilon_safe(release, "ball")
    eps_standard = extract_epsilon_safe(release, "standard")
    eps_primary = (
        eps_ball
        if mechanism == "ball"
        else eps_standard if mechanism == "standard" else float("inf")
    )

    return ReleaseBundle(
        name=name,
        mechanism=mechanism,
        release=release,
        noise_multiplier=float(noise_multiplier),
        epsilon_primary=float(eps_primary),
        epsilon_ball=float(eps_ball),
        epsilon_standard=float(eps_standard),
        utility_value=evaluate_accuracy_safe(release, X_public, y_public),
        recorder=recorder,
    )


def residualized_trace_for_trial(bundle: ReleaseBundle):
    metadata = make_trace_metadata_from_release(
        bundle.release,
        target_index=int(trial.target_index),
        extra={"mechanism": bundle.mechanism, "model_name": bundle.name},
    )

    reduction = (
        "sum"
        if str(
            bundle.release.training_config.get("normalize_noisy_sum_by", "batch_size")
        ).lower()
        == "none"
        else "mean"
    )

    trace = bundle.recorder.to_trace(
        state=bundle.release.extra.get("model_state", None),
        loss_name="binary_logistic",
        reduction=reduction,
        metadata=metadata,
    )

    residualized = subtract_known_batch_gradients(
        trace,
        ArrayDataset(trial.X_full, trial.y_full, name="attack_train"),
        target_index=int(trial.target_index),
        loss_name="binary_logistic",
        seed=0,
    )

    return trace, residualized

## Calibrate and train ERM, Ball-DP, and standard-DP releases

ERM is nonprivate and mainly serves as a utility and attack reference. The DP releases are calibrated to the same target privacy level.

In [ ]:
ball_noise = calibrate_noise_multiplier(privacy="ball_rdp")
standard_noise = calibrate_noise_multiplier(privacy="standard_rdp")

release_specs = [
    ("ERM", "erm", "noiseless", 0.0),
    ("Ball-DP", "ball", "ball_rdp", float(ball_noise)),
    ("Std-DP", "standard", "standard_rdp", float(standard_noise)),
]

bundles = []
for i, (name, mechanism, privacy, noise) in enumerate(release_specs):
    print(f"Training {name}...")
    bundle = train_recorded_release(
        name=name,
        mechanism=mechanism,
        privacy=privacy,
        noise_multiplier=float(noise),
        seed=int(SEED + 1000 * i),
    )
    bundles.append(bundle)

utility_rows = []
trace_cache = {}

for bundle in bundles:
    raw_trace, residual_trace = residualized_trace_for_trial(bundle)
    trace_cache[bundle.name] = (raw_trace, residual_trace)

    target_inclusions = sum(
        bool(np.any(np.asarray(step.batch_indices, dtype=np.int64) == trial.target_index))
        for step in raw_trace.steps
    )

    utility_rows.append(
        {
            "model": bundle.name,
            "mechanism": bundle.mechanism,
            "accuracy": bundle.utility_value,
            "noise_multiplier": bundle.noise_multiplier,
            "epsilon_primary": bundle.epsilon_primary,
            "epsilon_ball": bundle.epsilon_ball,
            "epsilon_standard": bundle.epsilon_standard,
            "retained_steps": len(raw_trace.steps),
            "target_inclusions": int(target_inclusions),
            "delta_ball": getattr(bundle.release.sensitivity, "delta_ball", np.nan),
            "delta_standard": getattr(bundle.release.sensitivity, "delta_std", np.nan),
        }
    )

utility_df = pd.DataFrame(utility_rows)
display(utility_df)

## ReRo bounds

We report two kinds of bounds:

1. a finite exact-ID point at $\kappa=1/m$ for the actual attack support;
2. continuous curves over $\kappa$ for a uniform ball prior, useful for visualizing how reconstruction bounds scale with prior concentration.

Known-inclusion attacks should be compared to revealed-transcript bounds. Unknown-inclusion attacks should be compared to hidden-transcript bounds. The RDP curve is a separate certificate and should not be confused with the revealed-inclusion observation model.

In [ ]:
finite_prior = make_finite_identification_prior(
    trial.support.X.reshape(trial.support.m, -1),
    weights=trial.support.weights,
)

finite_bound_rows = []
for bundle in bundles:
    if bundle.mechanism == "erm":
        continue
    for mode, label in [
        ("ball_sgd_hayes", "known/revealed"),
        ("ball_sgd_hidden", "unknown/hidden"),
        ("rdp", "RDP conversion"),
    ]:
        try:
            report = ball_rero(
                bundle.release,
                prior=finite_prior,
                eta_grid=(0.5,),
                mode=mode,
            )
            p = report.points[0]
            finite_bound_rows.append(
                {
                    "model": bundle.name,
                    "mode": label,
                    "raw_mode": mode,
                    "kappa": float(p.kappa),
                    "gamma_ball": float(p.gamma_ball),
                    "gamma_standard": (
                        np.nan if p.gamma_standard is None else float(p.gamma_standard)
                    ),
                }
            )
        except Exception as exc:
            finite_bound_rows.append(
                {
                    "model": bundle.name,
                    "mode": label,
                    "raw_mode": mode,
                    "kappa": float(trial.support.oblivious_kappa),
                    "gamma_ball": np.nan,
                    "gamma_standard": np.nan,
                    "error": str(exc),
                }
            )

finite_bound_df = pd.DataFrame(finite_bound_rows)
display(finite_bound_df)

In [ ]:
def eta_for_uniform_ball_kappa(radius: float, dimension: int, kappa: float) -> float:
    return float(radius * (float(kappa) ** (1.0 / float(dimension))))


uniform_prior = make_uniform_ball_prior(
    center=np.zeros(FEATURE_DIM, dtype=np.float32),
    radius=float(RADIUS),
    dimension=int(FEATURE_DIM),
)

kappa_grid = np.geomspace(1e-4, 0.95, 80)
eta_grid = tuple(
    eta_for_uniform_ball_kappa(RADIUS, FEATURE_DIM, float(k))
    for k in kappa_grid
)

curve_rows = []
for bundle in bundles:
    if bundle.mechanism == "erm":
        continue
    for mode, label in [
        ("ball_sgd_hayes", "known/revealed"),
        ("ball_sgd_hidden", "unknown/hidden"),
        ("rdp", "RDP conversion"),
    ]:
        try:
            report = ball_rero(
                bundle.release,
                prior=uniform_prior,
                eta_grid=eta_grid,
                mode=mode,
            )
            for point in report.points:
                curve_rows.append(
                    {
                        "model": bundle.name,
                        "mode": label,
                        "kappa": float(point.kappa),
                        "gamma_ball": float(point.gamma_ball),
                        "gamma_standard": (
                            np.nan
                            if point.gamma_standard is None
                            else float(point.gamma_standard)
                        ),
                    }
                )
        except Exception as exc:
            print(f"Skipping curve {bundle.name} {mode}: {exc}")

curve_df = pd.DataFrame(curve_rows)
display(curve_df.head())

## Run known-inclusion and unknown-inclusion finite-prior transcript attacks

Both attacks use the same fixed support $S$ and the same hidden target. Only the transcript likelihood changes.

In [ ]:
attack_rows = []

for bundle in bundles:
    raw_trace, residual_trace = trace_cache[bundle.name]

    for mode in ["known_inclusion", "unknown_inclusion"]:
        cfg = BallTraceMapAttackConfig(
            mode=mode,
            step_mode=("present_steps" if mode == "known_inclusion" else "all"),
            seed=int(SEED),
        )

        try:
            attack = attack_nonconvex_finite_prior_trial(
                residual_trace,
                trial,
                cfg=cfg,
                known_label=int(trial.support.center_y),
                eta_grid=(0.25, 0.50, 1.00),
            )

            attack_rows.append(
                {
                    "model": bundle.name,
                    "mode": mode,
                    "status": attack.status,
                    "accuracy": bundle.utility_value,
                    "epsilon_primary": bundle.epsilon_primary,
                    "baseline_kappa": trial.support.oblivious_kappa,
                    "source_exact_id": attack.metrics.get(
                        "source_exact_identification_success", np.nan
                    ),
                    "feature_exact_id": attack.metrics.get(
                        "exact_identification_success", np.nan
                    ),
                    "prior_rank": attack.metrics.get("prior_rank", np.nan),
                    "predicted_prior_index": attack.diagnostics.get(
                        "predicted_prior_index"
                    ),
                    "true_prior_index": attack.diagnostics.get("true_prior_index"),
                    "predicted_source_id": attack.diagnostics.get("predicted_source_id"),
                    "target_source_id": attack.diagnostics.get("target_source_id"),
                    "selected_step_count": attack.diagnostics.get("selected_step_count"),
                }
            )
        except Exception as exc:
            attack_rows.append(
                {
                    "model": bundle.name,
                    "mode": mode,
                    "status": "error",
                    "accuracy": bundle.utility_value,
                    "epsilon_primary": bundle.epsilon_primary,
                    "baseline_kappa": trial.support.oblivious_kappa,
                    "source_exact_id": np.nan,
                    "feature_exact_id": np.nan,
                    "error": str(exc),
                }
            )

attack_df = pd.DataFrame(attack_rows)
display(attack_df)

## Visualize utility, bounds, and attacks

The left plot shows the privacy-utility part of the training cycle. The middle plot shows known, hidden, and RDP reconstruction curves. The right plot compares empirical exact-ID success to the prior baseline.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.5), constrained_layout=True)

# Utility.
x = np.arange(len(utility_df))
colors = [MODEL_COLORS.get(m, "tab:gray") for m in utility_df["model"]]
axes[0].bar(x, utility_df["accuracy"], alpha=0.88, color=colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(utility_df["model"])
axes[0].set_ylim(0.0, 1.05)
axes[0].set_ylabel(r"public accuracy")
axes[0].set_title(r"Utility at matched privacy target")
for i, row in enumerate(utility_df.itertuples()):
    eps_text = r"$\varepsilon=\infty$" if not np.isfinite(row.epsilon_primary) else rf"$\varepsilon={row.epsilon_primary:.2f}$"
    sigma_text = rf"$\sigma={row.noise_multiplier:.2f}$"
    axes[0].text(i, min(1.02, row.accuracy + 0.045), eps_text + "\n" + sigma_text, ha="center", va="bottom", fontsize=9)

# Bound curves for the representative Ball-DP release.
if not curve_df.empty:
    sub = curve_df[curve_df["model"] == "Ball-DP"]
    axes[1].plot(kappa_grid, kappa_grid, linestyle="--", color="black", linewidth=1.3, label=r"baseline $\kappa$")
    for mode, mode_df in sub.groupby("mode"):
        mode_df = mode_df.sort_values("kappa")
        axes[1].plot(
            mode_df["kappa"],
            mode_df["gamma_ball"],
            linewidth=2.0,
            label=mode,
        )
    axes[1].set_xscale("log")
    axes[1].set_xlabel(r"prior mass $\kappa$")
    axes[1].set_ylabel(r"bound value $\Gamma(\kappa)$")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title(r"Ball-DP ReRo curves")
    axes[1].legend(loc="lower right")
else:
    axes[1].text(0.5, 0.5, "No bound curves", ha="center", va="center")
    axes[1].set_axis_off()

# Attacks.
plot_df = attack_df.copy()
plot_df["mode_label"] = plot_df["mode"].map(
    {"known_inclusion": "known", "unknown_inclusion": "unknown"}
)
x_labels = [f"{r.model}\n{r.mode_label}" for r in plot_df.itertuples()]
x = np.arange(len(plot_df))
colors = [MODEL_COLORS.get(m, "tab:gray") for m in plot_df["model"]]
axes[2].bar(x, plot_df["source_exact_id"].fillna(0.0), alpha=0.88, color=colors)
axes[2].axhline(
    trial.support.oblivious_kappa,
    linestyle="--",
    color="black",
    linewidth=1.3,
    label=rf"chance $\kappa=1/m={trial.support.oblivious_kappa:.3f}$",
)
axes[2].set_xticks(x)
axes[2].set_xticklabels(x_labels, rotation=30, ha="right")
axes[2].set_ylim(0.0, 1.05)
axes[2].set_ylabel(r"exact-ID success")
axes[2].set_title(r"Single finite-prior transcript attack")
axes[2].legend(loc="upper right")

plt.show()


## Training curves

The public evaluation history is useful for checking whether attack results are caused by a meaningful trained model or by undertraining.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 4.6), constrained_layout=True)

for bundle in bundles:
    hist = [
        dict(row)
        for row in get_public_curve_history(bundle.release)
        if "public_eval_accuracy" in row
    ]
    if not hist:
        continue
    hist = sorted(hist, key=lambda row: int(row["step"]))
    ax.plot(
        [int(row["step"]) for row in hist],
        [float(row["public_eval_accuracy"]) for row in hist],
        marker="o",
        linewidth=2.0,
        color=MODEL_COLORS.get(bundle.name, None),
        label=bundle.name,
    )

ax.set_xlabel(r"training step $t$")
ax.set_ylabel(r"public accuracy")
ax.set_ylim(0.0, 1.05)
ax.set_title(r"Public utility during training")
ax.legend(loc="lower right")
plt.show()


## Reusing this notebook on real data

To use real data, replace only the synthetic data cell with arrays from your experiment:

```python
X_train = ...
y_train = ...
X_public = ...
y_public = ...
```

Then keep the canonical setup:

```python
public_source = CandidateSource("public", X_public, y_public)
bank = find_feasible_replacement_banks(...)
support = select_support_from_bank(...)
trial = make_replacement_trial(...)
```

The same `trial` object can be passed to both convex and nonconvex attack wrappers. The observation model determines which scorer and which ReRo curve are appropriate.